# Data Feasibility Testing for Methodology
This notebook tests the availability and structure of the datasets and checks whether the proposed methods are feasible. The results are preliminary and are not intended to answer the final research questions.

## Testing

### Test 1 - Reporting channel distribution

Check which reporting channels are available in the 2025 dataset and how many requests were submitted through each channel.

In [1]:
import pandas as pd
import requests

url = "https://cos-data.seattle.gov/resource/5ngg-rpne.json"

params = {
    "$select": "methodreceivedname, count(*) as n",
    "$where": (
        "createddate >= '2025-01-01T00:00:00' "
        "AND createddate < '2026-01-01T00:00:00'"
    ),
    "$group": "methodreceivedname",
    "$order": "n DESC",
    "$limit": 100
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

channels = pd.DataFrame(response.json())
channels["n"] = pd.to_numeric(channels["n"])

display(channels)

,methodreceivedname,n
0,Find It Fix It Apps,268094
1,Citizen Web Intake App,36487
2,PremierOne CSR Mobile Apps,12990
3,Phone,10416
4,E-Mail,5054
5,Mass Entry,2873
6,Voice Mail,1980
7,Mail,30
8,Kiosk,3
9,Other,3


### Test 2 - Geographic data completness

Check how many requests from each reporting channel have latitude and longitude data.

In [2]:
params = {
    "$select": (
        "methodreceivedname, "
        "count(*) as total, "
        "count(latitude) as with_latitude, "
        "count(longitude) as with_longitude"
    ),
    "$where": (
        "createddate >= '2025-01-01T00:00:00' "
        "AND createddate < '2026-01-01T00:00:00'"
    ),
    "$group": "methodreceivedname",
    "$order": "total DESC",
    "$limit": 100
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

geo_check = pd.DataFrame(response.json())

for col in ["total", "with_latitude", "with_longitude"]:
    geo_check[col] = pd.to_numeric(geo_check[col])

geo_check["latitude_available_pct"] = (
    geo_check["with_latitude"] / geo_check["total"] * 100
).round(1)

display(geo_check)

,methodreceivedname,total,with_latitude,with_longitude,latitude_available_pct
0,Find It Fix It Apps,268094,268036,268036,100.0
1,Citizen Web Intake App,36487,34194,34194,93.7
2,PremierOne CSR Mobile Apps,12990,12987,12987,100.0
3,Phone,10416,9706,9706,93.2
4,E-Mail,5054,4375,4375,86.6
5,Mass Entry,2873,2872,2872,100.0
6,Voice Mail,1980,1733,1733,87.5
7,Mail,30,25,25,83.3
8,Kiosk,3,0,0,0.0
9,Other,3,3,3,100.0


### Test 3 - Request type distribution

Examine which request types are recorded through the app and phone channels.

In [3]:
params = {
    "$select": (
        "methodreceivedname, "
        "webintakeservicerequests, "
        "count(*) as n"
    ),
    "$where": (
        "createddate >= '2025-01-01T00:00:00' "
        "AND createddate < '2026-01-01T00:00:00' "
        "AND methodreceivedname IN ('Find It Fix It Apps', 'Phone')"
    ),
    "$group": "methodreceivedname, webintakeservicerequests",
    "$order": "n DESC",
    "$limit": 100
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

request_types = pd.DataFrame(response.json())

request_types["n"] = pd.to_numeric(request_types["n"])

display(request_types)

,methodreceivedname,webintakeservicerequests,n
0,Find It Fix It Apps,Unauthorized Encampment,64192
1,Find It Fix It Apps,Abandoned Vehicle/72hr Parking Ordinance,51002
2,Find It Fix It Apps,Illegal Dumping / Needles,26579
3,Find It Fix It Apps,Parking Enforcement,22715
4,Find It Fix It Apps,Graffiti,21726
...,...,...,...
63,Phone,Feedback about the Customer Service Requests P...,2
64,Phone,Animal Noise,2
65,Find It Fix It Apps,Business Related Complaint,2
66,Phone,Nuisance Dogs in a Park,1


### Test 4 - App vs. phone comparison

Compare app and phone request counts within the same request types to identify categories suitable for further analysis.

In [4]:
comparison = request_types.pivot_table(
    index="webintakeservicerequests",
    columns="methodreceivedname",
    values="n",
    aggfunc="sum",
    fill_value=0
)

comparison = comparison.rename(columns={
    "Find It Fix It Apps": "App",
    "Phone": "Phone"
})

comparison["Total"] = comparison["App"] + comparison["Phone"]

comparison = comparison.sort_values(
    "Total", ascending=False
)

display(comparison.head(20))

methodreceivedname,App,Phone,Total
webintakeservicerequests,,,
Unauthorized Encampment,64192,163,64355
Abandoned Vehicle/72hr Parking Ordinance,51002,4067,55069
Illegal Dumping / Needles,26579,372,26951
Parking Enforcement,22715,31,22746
Graffiti,21726,376,22102
General Inquiry - Customer Service Bureau,14081,1745,15826
Parks and Recreation Maintenance,12288,98,12386
Scooter or Bike Share Issue,8711,419,9130
Pothole,8205,12,8217


### Test 5 - Census tract boundaries

Load Seattle's 2020 census tract boundaries and check the geographic identifiers and coordinate systems.

In [8]:
%pip install geopandas
import geopandas as gpd

tract_url = (
    "https://data-seattlecitygis.opendata.arcgis.com/"
    "api/download/v1/items/9075e8c912a24c4b9458af8866c72ae7/"
    "geojson?layers=3"
)

tracts = gpd.read_file(tract_url)

print("Number of tracts:", len(tracts))
print("Coordinate system:", tracts.crs)
print("Available columns:")
print(tracts.columns.tolist())

display(tracts.head(3))

Note: you may need to restart the kernel to use updated packages.
Number of tracts: 177
Coordinate system: EPSG:2926
Available columns:
['OBJECTID', 'GEOID20', 'GROSS_ACRES', 'LAND_ACRES', 'WATER_ACRES', 'NAME', 'TRACT_NUMB', 'BASENAME', 'UVDA_AREA', 'CRA_NO', 'CRA_GRP', 'GEN_ALIAS', 'DETL_NAMES', 'C_DISTRICT', 'geometry']


,OBJECTID,GEOID20,GROSS_ACRES,LAND_ACRES,WATER_ACRES,NAME,TRACT_NUMB,BASENAME,UVDA_AREA,CRA_NO,CRA_GRP,GEN_ALIAS,DETL_NAMES,C_DISTRICT,geometry
0,1,53033000402,444.493643,444.527261,0.000000,Census Tract 4.02,402,4.02,Bitter Lake Village,9.1,9,Broadview/Bitter Lake,"Broadview, Bitter Lake",5,"POLYGON ((1267598.433 267477.138, 1267788.924 ..."
1,2,53033000403,180.930716,165.408389,15.535762,Census Tract 4.03,403,4.03,Bitter Lake Village,9.1,9,Broadview/Bitter Lake,"Broadview, Bitter Lake",5,"POLYGON ((1265848.993 271492.767, 1265842.812 ..."
2,3,53033000700,319.838996,319.862596,0.000000,Census Tract 7,700,7,Lake City,8.2,8,Olympic Hills/Victory Heights,"Lake City, Victory Heights, Olympic Hills, Pin...",5,"POLYGON ((1277629.018 265876.292, 1277833.975 ..."


### Test 6 - Sample service requests

Check a small sample of app and phone requests with geographic coordinates to test spatial matching.

In [9]:
import requests
import pandas as pd

url = "https://cos-data.seattle.gov/resource/5ngg-rpne.json"

samples = []

for channel in ["Find It Fix It Apps", "Phone"]:
    params = {
        "$select": (
            "servicerequestnumber, methodreceivedname, "
            "latitude, longitude, webintakeservicerequests"
        ),
        "$where": (
            "createddate >= '2025-01-01T00:00:00' "
            "AND createddate < '2026-01-01T00:00:00' "
            f"AND methodreceivedname = '{channel}' "
            "AND latitude IS NOT NULL "
            "AND longitude IS NOT NULL"
        ),
        "$order": "createddate DESC",
        "$limit": 150
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    samples.extend(response.json())

sample_df = pd.DataFrame(samples)

sample_df["latitude"] = pd.to_numeric(sample_df["latitude"])
sample_df["longitude"] = pd.to_numeric(sample_df["longitude"])

print("Sample size:", len(sample_df))
display(sample_df.groupby("methodreceivedname").size())
display(sample_df.head())

Sample size: 300


methodreceivedname
Find It Fix It Apps    150
Phone                  150
dtype: int64

,servicerequestnumber,methodreceivedname,latitude,longitude,webintakeservicerequests
0,25-00401651,Find It Fix It Apps,47.607158,-122.339803,General Inquiry - Customer Service Bureau
1,25-00401650,Find It Fix It Apps,47.662963,-122.326958,Abandoned Vehicle/72hr Parking Ordinance
2,25-00401649,Find It Fix It Apps,47.617608,-122.322247,Unauthorized Encampment
3,25-00401647,Find It Fix It Apps,47.623288,-122.318614,Unauthorized Encampment
4,25-00401646,Find It Fix It Apps,47.656847,-122.345213,Abandoned Vehicle/72hr Parking Ordinance


### Test 7 - Spatial join feasibility

Test whether service request locations can be matched to 2020 census tracts using GeoPandas.

In [10]:
# Convert latitude and longitude to geographic points
points = gpd.GeoDataFrame(
    sample_df,
    geometry=gpd.points_from_xy(
        sample_df["longitude"],
        sample_df["latitude"]
    ),
    crs="EPSG:4326"
)

# Convert to the same coordinate system as census tracts
points = points.to_crs(tracts.crs)

# Keep the tract ID and geographic boundaries
tract_boundaries = tracts[["GEOID20", "geometry"]].copy()
tract_boundaries["GEOID20"] = tract_boundaries["GEOID20"].astype(str)

# Spatial join
joined = gpd.sjoin(
    points,
    tract_boundaries,
    how="left",
    predicate="intersects"
)

# Check the results
print("Original reports:", len(points))
print("Joined rows:", len(joined))
print("Matched rows:", joined["GEOID20"].notna().sum())

print(
    "Match rate:",
    round(joined["GEOID20"].notna().mean() * 100, 2),
    "%"
)

display(
    joined[
        ["servicerequestnumber", "methodreceivedname", "GEOID20"]
    ].head(10)
)

Original reports: 300
Joined rows: 300
Matched rows: 296
Match rate: 98.67 %


,servicerequestnumber,methodreceivedname,GEOID20
0,25-00401651,Find It Fix It Apps,53033008101
1,25-00401650,Find It Fix It Apps,53033005201
2,25-00401649,Find It Fix It Apps,53033007405
3,25-00401647,Find It Fix It Apps,53033007501
4,25-00401646,Find It Fix It Apps,53033005000
5,25-00401645,Find It Fix It Apps,53033011200
6,25-00401642,Find It Fix It Apps,53033008002
7,25-00401641,Find It Fix It Apps,53033008002
8,25-00401640,Find It Fix It Apps,NaN
9,25-00401636,Find It Fix It Apps,53033009200


### Test 8 - ACS population and income fields

Check whether the ACS dataset includes total population and median household income variables needed for the planned analysis.

In [11]:
import requests
import pandas as pd

acs_url = (
    "https://services.arcgis.com/ZOyb2t4B0UYuYNYH/"
    "arcgis/rest/services/CensusTractTop50_5YALL/"
    "FeatureServer/0"
)

response = requests.get(
    acs_url,
    params={"f": "json"},
    timeout=60
)
response.raise_for_status()

metadata = response.json()

if "error" in metadata:
    raise RuntimeError(metadata["error"])

fields = pd.DataFrame(metadata["fields"])

print("Total fields:", len(fields))

# Search for population-related fields
population_fields = fields[
    fields["name"].str.contains(
        "population|pop_|totalpop",
        case=False, na=False
    ) |
    fields["alias"].str.contains(
        "population",
        case=False, na=False
    )
]

print("\nPOPULATION FIELDS:")
display(population_fields[["name", "alias"]])

# Search for income-related fields
income_fields = fields[
    fields["name"].str.contains(
        "income|b19013|medh",
        case=False, na=False
    ) |
    fields["alias"].str.contains(
        "income",
        case=False, na=False
    )
]

print("\nINCOME FIELDS:")
display(income_fields[["name", "alias"]])

Total fields: 128

POPULATION FIELDS:


,name,alias
10,TOTAL_POPULATION,Total population
11,Age_Under_18,Population under 18
12,PCT_POP_UNDER_18,Percent of Population under 18 years
13,AGE_18_AND_OVER,Population 18 Years and over
14,PCT_POP_18_OVER,Percent of Population 18 years and over
15,Age_65_and_over,Population 65 Years and over
16,PCT_POP_65_OVER,Percent of Population 65 years and over
35,PCT_PERSON_OF_COLOR,Percent of Population of Color
37,POPULATION_IN_HOUSEHOLDS,Population in Households
38,PCT_POP_IN_HH,Percent Population in Households



INCOME FIELDS:


,name,alias
81,WITH_MORTGAGE_SMOCAPI_COMPUTED,Housing units with a mortgage where selected m...
82,WITH_MORTGAGE_SMOCAPI_LESS_20,Selected Monthly Owner Costs as a Percentage o...
83,WITHMORTGAGESMOCAPI_20_TO_24_9,Selected Monthly Owner Costs as a Percentage o...
84,WITHMORTGAGESMOCAPI_25_TO_29_9,Selected Monthly Owner Costs as a Percentage o...
85,WITHMORTGAGESMOCAPI_30_TO_34_9,Selected Monthly Owner Costs as a Percentage o...
86,WITHMORTGAGESMOCAPI_35_OR_MORE,Selected Monthly Owner Costs as a Percentage o...
87,WITHMORTGAGESMOCAPINOTCOMPUTED,Housing units with a mortgage where selected m...
88,PERC_SMOCAPI_WMORT_30_OR_MORE,Percent of Owner-occupied housing units with a...
90,RENT_GRAPI_COMPUTED,Occupied units paying rent where gross rent as...
91,GRAPI_LESS_THAN_15_0,Gross Rent as a Percentage of Household Income...


### Test 9 - Acs geographic identifiers

Check the available census tract identifiers and year fields to determine whether the ACS data can be matched with the 2020 census tract boundaries.

In [12]:
# Find geographic identifier and year fields
id_year_fields = fields[
    fields["name"].str.contains(
        "geoid|geoid20|tract|vintage|year",
        case=False, na=False
    ) |
    fields["alias"].str.contains(
        "geoid|tract|vintage|year",
        case=False, na=False
    )
]

print("GEOGRAPHIC ID & YEAR FIELDS:")
display(id_year_fields[["name", "alias"]])

# Inspect a few records
params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "resultRecordCount": 3,
    "f": "json"
}

response = requests.get(
    acs_url + "/query",
    params=params,
    timeout=60
)
response.raise_for_status()

data = response.json()

if "error" in data:
    raise RuntimeError(data["error"])

sample_acs = pd.DataFrame([
    feature["attributes"]
    for feature in data["features"]
])

print("Sample records:", len(sample_acs))
display(sample_acs[id_year_fields["name"].tolist()])

GEOGRAPHIC ID & YEAR FIELDS:


,name,alias
1,GEOID,GEOID
2,NAME,Tract Name
3,TRACT_LABEL,Tract Label
12,PCT_POP_UNDER_18,Percent of Population under 18 years
13,AGE_18_AND_OVER,Population 18 Years and over
14,PCT_POP_18_OVER,Percent of Population 18 years and over
15,Age_65_and_over,Population 65 Years and over
16,PCT_POP_65_OVER,Percent of Population 65 years and over
17,MEDIAN_AGE,Median Age (years)
61,Population_25_years_and_over,Population 25 years and over


Sample records: 3


,GEOID,NAME,TRACT_LABEL,PCT_POP_UNDER_18,AGE_18_AND_OVER,PCT_POP_18_OVER,Age_65_and_over,PCT_POP_65_OVER,MEDIAN_AGE,Population_25_years_and_over,POPULATION_5_YEARS_AND_OVER,Population_16_and_over,ACS_VINTAGE
0,53033001500,Census Tract 15,15,19.3,1935,80.7,404,16.8,46.9,1827,2301,1961,5Y15
1,53033001600,Census Tract 16,16,18.4,3355,81.6,751,18.3,45.0,3219,3850,3418,5Y15
2,53033020100,Census Tract 201,201,18.0,2564,82.0,931,29.8,54.4,2470,3012,2660,5Y15


### Test 10 - ACS data vintage

Check the availble ACS data years and identify a suitable version for the 2025 service request analysis.

In [14]:
params = {
    "where": "1=1",
    "outFields": "ACS_VINTAGE",
    "returnGeometry": "false",
    "returnDistinctValues": "true",
    "orderByFields": "ACS_VINTAGE",
    "f": "json"
}

response = requests.get(
    acs_url + "/query",
    params=params,
    timeout=60
)
response.raise_for_status()

data = response.json()

if "error" in data:
    raise RuntimeError(data["error"])

vintages = pd.DataFrame([
    feature["attributes"]
    for feature in data["features"]
])

display(vintages)

,ACS_VINTAGE
0,5Y10
1,5Y15
2,5Y20
3,5Y24


## AI Use

I used ChatGPT to help generate Python code for the data feasibility tests in this notebook. I explained what I wanted to test, and ChatGPT provided the corresponding code. I ran the code and reviewed the outputs to check whether the datasets and methods could support my research.

### Example Prompts:
- I want to check the reporting channels in Seattle's Customer Service Requests dataset. Filter the data to 2025, group by methodreceivedname, and count how many reports are in each channel. I mainly want to see if App and Phone can be compared. Python code.
- I want to test spatial join with GeoPandas. Get 150 App reports and 150 Phone reports with coordinates, convert them into geographic points, and match them with Seattle's 2020 census tract boundaries. Then calculate how many records matched successfully and the match rate. Python code.